In [0]:
# Configuração do projeto
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "gep_public_procurement"
VOLUME = "arquivos"

# Protege os identificadores usados no SQL
def sql_identifier(nome):
    return "`" + nome.replace("`", "``") + "`"

SCHEMA_SQL = (
    f"{sql_identifier(CATALOG)}."
    f"{sql_identifier(SCHEMA)}"
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_SQL}")

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{SCHEMA_SQL}.{sql_identifier(VOLUME)}"
)

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print("Catálogo:", CATALOG)
print("Schema:", SCHEMA)
print("Destino dos arquivos:", BASE_PATH)

Catálogo: workspace
Schema: gep_public_procurement
Destino dos arquivos: /Volumes/workspace/gep_public_procurement/arquivos


In [0]:
catalogo = (
    spark.read
    .option("header", True)
    .schema("""
        codigoItemCatalogo STRING,
        descricaoItem STRING,
        nomeUnidadeFornecimento STRING,
        siglaUnidadeMedida STRING,
        unidade_canonica STRING
    """)
    .csv(f"{BASE_PATH}/public_procurement_catalog_items.csv")
)

conversoes = (
    spark.read
    .option("header", True)
    .schema("""
        source_sigla STRING,
        canonical_unit STRING,
        conversion_factor DECIMAL(18,9)
    """)
    .csv(f"{BASE_PATH}/public_procurement_unit_conversions.csv")
)

total_itens = catalogo.count()
total_conversoes = conversoes.count()

print("Itens do catálogo:", total_itens)
print("Conversões de unidades:", total_conversoes)

assert total_itens == 20, "Esperados 20 itens no catálogo."
assert total_conversoes == 7, "Esperadas 7 conversões."

display(catalogo)
display(conversoes)

Itens do catálogo: 20
Conversões de unidades: 7


codigoItemCatalogo,descricaoItem,nomeUnidadeFornecimento,siglaUnidadeMedida,unidade_canonica
217129,"BISCOITO, APRESENTACAO: QUADRADO , SABOR: AGUA E SAL , CLASSIFICACAO: SALGADO , CARACTERISTICAS ADICIONAIS: SEM RECHEIO",PACOTE,G,KG
226093,"SACO PLASTICO LIXO, CAPACIDADE: 50 L, COR: PRETA , APRESENTACAO: PECA UNICA , LARGURA: 63 CM, ALTURA: 80 CM",PACOTE,UN,UN
226094,"SACO PLASTICO LIXO, CAPACIDADE: 100 L, COR: PRETA , APRESENTACAO: PECA UNICA , LARGURA: 80 CM, ALTURA: 100 CM",PACOTE,UN,UN
237589,"COADOR DESCARTAVEL CAFE, MATERIAL: PAPEL , TAMANHO: 102",CAIXA,UN,UN
237928,"CIMENTO PORTLAND, MATERIAL: CLINKER , TIPO: CP II",SACO,KG,KG
242775,"CHA ALIMENTACAO, TIPO: CHA DE CAMOMILA , USO: ALIMENTICIO",CAIXA,UN,UN
244498,"FAROFA PRONTA, TIPO: TRADICIONAL , SABOR: NATURAL , PRAZO VALIDADE: 4 MES, QUANTIDADE CALORIAS: 210 KCAL/G, CARACTERISTICAS ADICIONAIS: COM FARINHA DE MADIOCA TEMPERADA",PACOTE,G,KG
245189,"CHA ALIMENTACAO, TIPO: ERVA DOCE , USO: ALIMENTICIO",CAIXA,UN,UN
255136,"SABAO PO, APLICACAO: LAVAR ROUPAS , ADITIVOS: ALVEJANTE , ODOR: FLORAL , ASPECTO FISICO: PO , COMPOSICAO: AGUA, ALQUIL BENZENO SULFATO DE SODIO, CORANTE, CA , CARACTERISTICAS ADICIONAIS: BIODEGRADAVEL",PACOTE,KG,KG
257647,"GIZ CERA, MATERIAL: CERA PLASTICA COM CORANTE ATOXICO , COR: VARIADAS , TAMANHO: GRANDE , ESPESSURA: GROSSA",CAIXA,UN,UN


source_sigla,canonical_unit,conversion_factor
UN,UN,1.000000000
PC,UN,1.000000000
FL,UN,1.000000000
KG,KG,1.000000000
G,KG,0.001000000
L,L,1.000000000
ML,L,0.001000000


In [0]:
PREFIX = "workspace.gep_public_procurement"

(
    catalogo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.ref_catalogo")
)

(
    conversoes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.ref_conversoes")
)

print("Tabelas de referência salvas.")

Tabelas de referência salvas.


In [0]:
import json
import uuid
import requests

from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)

URL = (
    "https://dadosabertos.compras.gov.br"
    "/modulo-pesquisa-preco/1_consultarMaterial"
)

RUN_ID = str(uuid.uuid4())

codigos = [
    row["codigoItemCatalogo"]
    for row in (
        catalogo
        .select("codigoItemCatalogo")
        .distinct()
        .orderBy("codigoItemCatalogo")
        .collect()
    )
]

linhas_bronze = []
logs = []

print("Execução:", RUN_ID)
print("Códigos a consultar:", len(codigos))

with requests.Session() as session:
    for codigo in codigos:
        parametros = {
            "tipo": "codigoItemCatalogo",
            "codigo": codigo,
            "pagina": 1,
            "tamanhoPagina": 100,
        }

        instante = datetime.now(timezone.utc)

        url_consulta = requests.Request(
            "GET", URL, params=parametros
        ).prepare().url

        status_http = None
        quantidade_recebida = 0
        resultado_consulta = "erro"
        detalhe = None

        try:
            resposta = session.get(
                URL,
                params=parametros,
                timeout=(10, 60),
            )

            status_http = resposta.status_code

            if status_http != 200:
                resultado_consulta = "erro_http"
                detalhe = resposta.text[:1000]

            else:
                payload = resposta.json()

                if not isinstance(payload, dict):
                    raise ValueError("A resposta não é um objeto JSON.")

                registros = payload.get("resultado")

                if not isinstance(registros, list):
                    raise ValueError(
                        "O campo 'resultado' está ausente ou não é uma lista."
                    )

                if not all(isinstance(item, dict) for item in registros):
                    raise ValueError(
                        "Há elementos em 'resultado' que não são objetos JSON."
                    )

                # Preserva todos os registros, inclusive os que serão
                # descartados posteriormente pelas regras da Silver.
                for registro in registros:
                    linhas_bronze.append((
                        RUN_ID,
                        str(codigo),
                        instante,
                        json.dumps(
                            registro,
                            ensure_ascii=False,
                            allow_nan=False,
                        ),
                    ))

                quantidade_recebida = len(registros)
                resultado_consulta = (
                    "sucesso" if registros else "sem_registros"
                )

        except requests.exceptions.JSONDecodeError as erro:
            resultado_consulta = "json_invalido"
            detalhe = str(erro)[:1000]

        except requests.exceptions.RequestException as erro:
            resultado_consulta = "erro_rede"
            detalhe = f"{type(erro).__name__}: {erro}"[:1000]

        except ValueError as erro:
            resultado_consulta = "resposta_invalida"
            detalhe = str(erro)[:1000]

        logs.append((
            RUN_ID,
            str(codigo),
            instante,
            url_consulta,
            status_http,
            resultado_consulta,
            quantidade_recebida,
            detalhe,
        ))

        print(
            f"Código {codigo} | HTTP {status_http} | "
            f"{resultado_consulta} | "
            f"{quantidade_recebida} registros"
        )


schema_log = StructType([
    StructField("run_id", StringType(), False),
    StructField("codigo_solicitado", StringType(), False),
    StructField("consultado_em", TimestampType(), False),
    StructField("url", StringType(), False),
    StructField("status_http", IntegerType(), True),
    StructField("resultado_consulta", StringType(), False),
    StructField("qtd_registros", IntegerType(), False),
    StructField("detalhe", StringType(), True),
])

log_df = spark.createDataFrame(logs, schema=schema_log)

# Mantém o histórico das tentativas de extração.
(
    log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{PREFIX}.log_extracao")
)

# Evita substituir uma Bronze existente por uma extração sem dados.
if not linhas_bronze:
    display(log_df)
    raise RuntimeError(
        "Nenhum registro foi extraído. "
        "Os logs foram salvos; a Bronze não foi alterada."
    )


schema_bronze = StructType([
    StructField("run_id", StringType(), False),
    StructField("codigo_solicitado", StringType(), False),
    StructField("extraido_em", TimestampType(), False),
    StructField("registro_json", StringType(), False),
])

bronze_df = spark.createDataFrame(
    linhas_bronze,
    schema=schema_bronze,
)

# Para este teste, a Bronze representa a extração mais recente.
# Uma nova execução substitui esse snapshot, sem acumular cópias.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.bronze_precos")
)

total_bronze = spark.table(f"{PREFIX}.bronze_precos").count()

assert total_bronze == sum(
    linha[6] for linha in logs
), "A contagem da Bronze difere da quantidade registrada nos logs."

falhas = [
    linha for linha in logs
    if linha[5] not in ("sucesso", "sem_registros")
]

print("\nRESUMO DA EXTRAÇÃO")
print("Execução:", RUN_ID)
print("Códigos consultados:", len(logs))
print("Consultas com falha:", len(falhas))
print("Registros na Bronze:", total_bronze)

if falhas:
    print(
        "ATENÇÃO: extração parcial. "
        "Confira os códigos com falha antes da entrega."
    )

display(log_df.orderBy("codigo_solicitado"))
display(spark.table(f"{PREFIX}.bronze_precos").limit(5))

Execução: a7ff4fb4-9ed6-485d-a4d0-3611b361f67b
Códigos a consultar: 20
Código 217129 | HTTP 200 | sucesso | 100 registros
Código 226093 | HTTP 200 | sucesso | 100 registros
Código 226094 | HTTP 200 | sucesso | 100 registros
Código 237589 | HTTP 200 | sucesso | 100 registros
Código 237928 | HTTP 200 | sucesso | 100 registros
Código 242775 | HTTP 200 | sucesso | 100 registros
Código 244498 | HTTP 200 | sucesso | 100 registros
Código 245189 | HTTP 200 | sucesso | 100 registros
Código 255136 | HTTP 200 | sucesso | 100 registros
Código 257647 | HTTP 200 | sucesso | 100 registros
Código 260613 | HTTP 200 | sucesso | 100 registros
Código 264210 | HTTP 200 | sucesso | 100 registros
Código 273052 | HTTP 200 | sucesso | 100 registros
Código 273455 | HTTP 200 | sucesso | 100 registros
Código 298548 | HTTP 200 | sucesso | 100 registros
Código 320920 | HTTP 200 | sucesso | 100 registros
Código 326278 | HTTP 200 | sucesso | 100 registros
Código 326284 | HTTP 200 | sucesso | 100 registros
Código 3723

run_id,codigo_solicitado,consultado_em,url,status_http,resultado_consulta,qtd_registros,detalhe
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,217129,2026-11-13T21:42:02.506Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=217129&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,226093,2026-11-13T21:42:04.229Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=226093&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,226094,2026-11-13T21:42:05.045Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=226094&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,237589,2026-11-13T21:42:05.527Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=237589&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,237928,2026-11-13T21:42:06.097Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=237928&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,242775,2026-11-13T21:42:06.500Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=242775&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,244498,2026-11-13T21:42:07.348Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=244498&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,245189,2026-11-13T21:42:07.934Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=245189&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,255136,2026-11-13T21:42:08.485Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=255136&pagina=1&tamanhoPagina=100,200,sucesso,100,null
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,257647,2026-11-13T21:42:09.042Z,https://dadosabertos.compras.gov.br/modulo-pesquisa-preco/1_consultarMaterial?tipo=codigoItemCatalogo&codigo=257647&pagina=1&tamanhoPagina=100,200,sucesso,100,null


run_id,codigo_solicitado,extraido_em,registro_json
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,260613,2026-11-13T21:42:09.428Z,"{""idCompra"": 92636305000192026, ""dataCompra"": ""2026-08-20"", ""forma"": ""SISRP"", ""modalidade"": 5, ""dataHoraAtualizacaoCompra"": ""2026-11-11T02:29:19.934509"", ""idItemCompra"": 13219844, ""numeroItemCompra"": 62, ""niFornecedor"": ""45135863000104"", ""codigoItemCatalogo"": 260613, ""quantidade"": 80.0, ""precoUnitario"": 12.2, ""descricaoItem"": ""TALHER DESCARTÁVEL, MATERIAL: PLÁSTICO , TIPO: MEXEDOR DE CAFEZINHO , APLICAÇÃO: COPA E COZINHA , COR: BRANCO TRANSPARENTE "", ""siglaUnidadeFornecimento"": ""PCT"", ""nomeUnidadeFornecimento"": ""PACOTE"", ""capacidadeUnidadeFornecimento"": 100.0, ""siglaUnidadeMedida"": ""UN"", ""nomeUnidadeMedida"": null, ""criterioJulgamento"": ""1"", ""percentualMaiorDesconto"": 0.0, ""nomeFornecedor"": ""MALTA COMERCIO LTDA"", ""marca"": ""STRAWPLAST"", ""dataResultado"": ""2026-11-10"", ""dataHoraAtualizacaoItem"": ""2026-11-11T07:44:06.39651"", ""codigoUasg"": ""926363"", ""nomeUasg"": ""CONSELHO REGIONAL DE ENGENHARIA E AGRONOMIA"", ""codigoOrgao"": 39006, ""nomeOrgao"": ""CONSELHO REG.DE ENGENHARIA E AGRONOMIA DO ES"", ""estado"": ""ES"", ""codigoMunicipio"": 3205309, ""municipio"": ""VITÓRIA"", ""poder"": ""E"", ""esfera"": ""F"", ""dataHoraAtualizacaoUasg"": ""2024-03-01T08:12:00"", ""codigoClasse"": 7340, ""nomeClasse"": ""CUTELARIA E TALHERES"", ""idCompraItem"": ""9263630500019202600062"", ""objetoCompra"": ""registro de preços para futura e eventual aquisição de materiais de expediente, papel sulfite e vergê, pilhas e materiais de copa/cozinha"", ""descricaoDetalhadaItem"": ""Talher Descartável material: plástico, tipo: mexedor de cafezinho, aplicação: copa e cozinha, cor: branco transparente "", ""dataAtualizacaoFato"": ""2026-11-11T07:44:06.39651"", ""codigoPdm"": ""12501"", ""nomePdm"": ""TALHER DESCARTÁVEL""}"
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,260613,2026-11-13T21:42:09.428Z,"{""idCompra"": 93000305000072026, ""dataCompra"": ""2026-08-11"", ""forma"": ""SISPP"", ""modalidade"": 5, ""dataHoraAtualizacaoCompra"": ""2026-11-10T02:30:05.053934"", ""idItemCompra"": 13208670, ""numeroItemCompra"": 18, ""niFornecedor"": ""44798994000109"", ""codigoItemCatalogo"": 260613, ""quantidade"": 550.0, ""precoUnitario"": 6.9, ""descricaoItem"": ""TALHER DESCARTÁVEL, MATERIAL: PLÁSTICO , TIPO: MEXEDOR DE CAFEZINHO , APLICAÇÃO: COPA E COZINHA , COR: BRANCO TRANSPARENTE "", ""siglaUnidadeFornecimento"": ""UN"", ""nomeUnidadeFornecimento"": ""UNIDADE"", ""capacidadeUnidadeFornecimento"": 0.0, ""siglaUnidadeMedida"": null, ""nomeUnidadeMedida"": null, ""criterioJulgamento"": ""1"", ""percentualMaiorDesconto"": 0.0, ""nomeFornecedor"": ""COMERCIAL SAO CARLOS DE PRODUTOS ALIMENTICIOS LTDA"", ""marca"": ""MEXEBEM"", ""dataResultado"": ""2026-11-09"", ""dataHoraAtualizacaoItem"": ""2026-11-10T07:11:33.660018"", ""codigoUasg"": ""930003"", ""nomeUasg"": ""FUNDAÇÃO HOSPITAL S.LYDIA DE RIBEIRÃO PRETO"", ""codigoOrgao"": 99833, ""nomeOrgao"": ""FUNDAÇÃO HOSPITAL S.LYDIA DE RIBEIRÃO PRETO"", ""estado"": ""SP"", ""codigoMunicipio"": 3543402, ""municipio"": ""RIBEIRÃO PRETO"", ""poder"": ""E"", ""esfera"": ""M"", ""dataHoraAtualizacaoUasg"": ""2024-10-13T11:09:00"", ""codigoClasse"": 7340, ""nomeClasse"": ""CUTELARIA E TALHERES"", ""idCompraItem"": ""9300030500007202600018"", ""objetoCompra"": ""Registro de Preços para futura aquisição de materiais descartáveis a serem utilizados na Fundação Hospital Santa Lydia, pelo período de 12 (doze) meses, podendo ser prorrogada por igual período com renovação total do quantitativo."", ""descricaoDetalhadaItem"": ""Talher Descartável Talher Descartável, Material:Plástico, Tipo:Mexedor De Cafezinho, Aplicação:Copa E Cozinha, Cor:Branco Transparente 18 - Mexedor descartável de café (500u)"", ""dataAtualizacaoFato"": ""2026-11-10T07:11:33.660018"", ""codigoPdm"": ""12501"", ""nomePdm"": ""TALHER DESCARTÁVEL""}"
a7ff4fb4-9ed6-485d-a4d0-3611b36